# Tome C2, Chapter 12: Approximation and Online Algorithms (Python companion)

Exercises first, solutions at the end. We put every guarantee of the chapter next to the truth, computed exactly on instances small enough for brute force: vertex cover by a matching and by the linear program, greedy set cover and its prices, the Christofides–Serdyukov tour against Held–Karp, the knapsack approximation scheme, three roads to MAX-SAT, the Goemans–Williamson cut with a home-made semidefinite solver, ski rental and paging, and multiplicative weights playing a game against itself.

*Before each solution, predict what the output will look like: is the worst ratio met in practice, or only on the cruel instances of the exercises?*

In [ ]:
import random, math, time, itertools
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import linprog, milp, LinearConstraint, Bounds
random.seed(12); rng = np.random.default_rng(12)

## Exercises

**P12.1 (★) Fire stations.** Implement the matching algorithm for vertex cover (Proposition 12.2) and the rounding of the linear program (Proposition 12.3). On random graphs compare both with the exact optimum (an integer program). Check that the optimal basic solutions of the linear program are half-integral (Exercise 12.2).

**P12.2 (★) Greedy prices.** Implement greedy set cover with the prices of Theorem 12.4. Check on random instances that the prices divided by $H_d$ are dual feasible, compare greedy with the linear program and the optimum, and run greedy on the tight family of Exercise 12.3.

**P12.3 (★★) Christofides–Serdyukov.** Implement the algorithm of Theorem 12.7 and the simpler doubled tree. Compare both with the optimum, computed by the Held–Karp dynamic program (Chapter 5), on random points of the plane.

**P12.4 (★) Rounding values.** Implement the approximation scheme of Theorem 12.8 and compare value and time with the exact dynamic program for several $\varepsilon$; add the greedy of Exercise 12.5.

**P12.5 (★★) Three roads to MAX-SAT.** On random formulas with clauses of mixed lengths, compare the random assignment (Proposition 12.9), its derandomisation by conditional expectations (Exercise 12.6), randomised rounding of the linear program, and the best of the two (Theorem 12.11), with the exact optimum.

**P12.6 (★★★) A random hyperplane.** Solve the semidefinite relaxation of MAX-CUT by a simple coordinate ascent on unit vectors, bound it from above by $\frac n4\lambda_{\max}(L)$, round with random hyperplanes (Theorem 12.14) and compare with the maximum cut found by brute force. Check Lemma 12.12 by simulation and compute the constant of Lemma 12.13. Treat the pentagon of Exercise 12.8.

**P12.7 (★) Skis and pages.** Compute the competitive ratio of buying on day $B$ against the worst season length (Proposition 12.15). Simulate LRU, FIFO and Belady's furthest-in-future rule on random and adversarial request sequences (Theorem 12.16, Exercise 12.9).

**P12.8 (★★) Experts.** Run Hedge on random and adversarial losses and compare its regret with the bound of Theorem 12.17; watch a deterministic follower lose (Exercise 12.10). Then let two Hedge players play a zero-sum game and compare the average payoff with the value given by the linear program of Exercise 7.7.

## Solutions

### P12.1 Fire stations

In [ ]:
def random_graph(n, p):
    return [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < p]

def vc_matching(n, E):
    C = set()
    for u, v in E:                       # a maximal matching, greedily
        if u not in C and v not in C:
            C |= {u, v}
    return C

def vc_lp(n, E):
    A = np.zeros((len(E), n))
    for k, (u, v) in enumerate(E):
        A[k, u] = A[k, v] = -1           # -x_u - x_v <= -1
    r = linprog(np.ones(n), A_ub=A, b_ub=-np.ones(len(E)), bounds=(0, 1), method="highs-ds")
    return r.x, r.fun

def vc_exact(n, E):
    A = np.zeros((len(E), n))
    for k, (u, v) in enumerate(E):
        A[k, u] = A[k, v] = 1
    r = milp(np.ones(n), constraints=LinearConstraint(A, lb=1), integrality=np.ones(n), bounds=Bounds(0, 1))
    return round(r.fun)

rows, halfint = [], True
for trial in range(200):
    n = random.randint(10, 40); E = random_graph(n, random.choice([0.1, 0.2, 0.4]))
    if not E: continue
    x, lpval = vc_lp(n, E)
    halfint &= bool(np.all(np.isclose(x[:, None], [0, .5, 1]).any(axis=1)))
    Cm = vc_matching(n, E); Cr = {v for v in range(n) if x[v] >= 0.5 - 1e-9}
    assert all(u in Cr or v in Cr for u, v in E)
    opt = vc_exact(n, E)
    rows.append((len(Cm) / opt, len(Cr) / opt, opt / lpval))
rows = np.array(rows)
print(f"{len(rows)} random graphs; all LP vertices half-integral: {halfint}")
print(f"matching / OPT: mean {rows[:,0].mean():.3f}, worst {rows[:,0].max():.3f}   (guarantee 2)")
print(f"LP rounding / OPT: mean {rows[:,1].mean():.3f}, worst {rows[:,1].max():.3f}   (guarantee 2)")
print(f"OPT / LP: mean {rows[:,2].mean():.3f}, worst {rows[:,2].max():.3f}")
n = 21; Kn = [(u, v) for u in range(n) for v in range(u + 1, n)]
x, lpval = vc_lp(n, Kn)
print(f"complete graph K_{n}: LP = {lpval}, OPT = {vc_exact(n, Kn)}, matching gives {len(vc_matching(n, Kn))}")

The simplex method of HiGHS (`highs-ds`) returns vertices of the polyhedron, and every one of them was half-integral, as Nemhauser and Trotter promise. On average both algorithms are well inside their factor $2$, but the factor is met: on the sparsest graphs, a few disjoint edges for instance, the matching algorithm takes both ends of every edge where one suffices. The complete graph shows where the factor $2$ comes from: the linear program puts $\frac12$ everywhere and gets $n/2$, while the optimum is $n-1$; any proof that compares with the linear program cannot do better than $2$.

### P12.2 Greedy prices

In [ ]:
def greedy_cover(U, sets, cost):
    covered, chosen, price = set(), [], {}
    while len(covered) < len(U):
        j = min((j for j in range(len(sets)) if sets[j] - covered),
                key=lambda j: cost[j] / len(sets[j] - covered))
        new = sets[j] - covered
        for e in new: price[e] = cost[j] / len(new)
        covered |= new; chosen.append(j)
    return chosen, price

def cover_lp(U, sets, cost, integral=False):
    A = np.array([[1.0 if e in S else 0.0 for S in sets] for e in U])
    if integral:
        r = milp(cost, constraints=LinearConstraint(A, lb=1), integrality=np.ones(len(sets)), bounds=Bounds(0, 1))
    else:
        r = linprog(cost, A_ub=-A, b_ub=-np.ones(len(U)), bounds=(0, None), method="highs")
    return r.fun

H = lambda d: sum(1 / i for i in range(1, d + 1))
worst, ok = 0, True
for trial in range(200):
    m, ns = 40, 25
    U = list(range(m))
    sets = [set(random.sample(U, random.randint(2, 10))) for _ in range(ns)]
    for e in U: sets[random.randrange(ns)].add(e)          # make sure a cover exists
    cost = np.array([random.uniform(1, 3) for _ in sets])
    chosen, price = greedy_cover(U, sets, cost)
    g = sum(cost[j] for j in chosen); d = max(map(len, sets))
    ok &= all(sum(price[e] for e in S) / H(d) <= cost[j] + 1e-9 for j, S in enumerate(sets))
    lp, opt = cover_lp(U, sets, cost), cover_lp(U, sets, cost, True)
    worst = max(worst, g / opt); assert g <= H(d) * lp + 1e-9
print(f"prices / H_d dual feasible in all 200 instances: {ok}; worst greedy/OPT = {worst:.3f} (H_10 = {H(10):.3f})")

for k in range(2, 9):
    N = 2**k - 1
    rows = [set(range(N)), set(range(N, 2 * N))]
    blocks = [set(range(2**j - 1, 2**(j + 1) - 1)) | set(range(N + 2**j - 1, N + 2**(j + 1) - 1)) for j in range(k)]
    sets = blocks[::-1] + rows                              # ties broken towards the blocks
    chosen, _ = greedy_cover(list(range(2 * N)), sets, np.ones(len(sets)))
    print(f"k = {k}: {2*N:4d} elements, greedy uses {len(chosen)} sets, OPT = 2")

The prices divided by $H_d$ never overpaid a set, which is the claim inside the proof of Theorem 12.4, and greedy stayed within a few per cent of the optimum on random instances. On the family of Exercise 12.3, with ties broken against us, greedy takes $k$ sets where two suffice: the ratio $k/2$ grows like $\frac12\log_2$ of the number of elements, and the logarithm in the theorem is really there.

### P12.3 Christofides–Serdyukov

In [ ]:
def held_karp(D):
    n = len(D); full = 1 << (n - 1)
    best = np.full((full, n - 1), np.inf)
    for j in range(n - 1): best[1 << j, j] = D[n - 1][j]   # start at city n-1
    for S in range(1, full):
        for j in range(n - 1):
            if not (S >> j) & 1 or best[S, j] == np.inf: continue
            for k in range(n - 1):
                if not (S >> k) & 1:
                    T = S | (1 << k)
                    c = best[S, j] + D[j][k]
                    if c < best[T, k]: best[T, k] = c
    return min(best[full - 1, j] + D[j][n - 1] for j in range(n - 1))

def shortcut(MG, n):
    tour, seen = [], set()
    for u, v in nx.eulerian_circuit(MG, source=0):
        if u not in seen: seen.add(u); tour.append(u)
    return tour + [tour[0]]

def tour_length(t, D): return sum(D[a][b] for a, b in zip(t, t[1:]))

def christofides(D):
    n = len(D); G = nx.complete_graph(n)
    for u, v in G.edges: G[u][v]["weight"] = D[u][v]
    T = nx.minimum_spanning_tree(G)
    odd = [v for v in T if T.degree(v) % 2]
    M = nx.min_weight_matching(G.subgraph(odd))
    MG = nx.MultiGraph(T); MG.add_edges_from(M)
    return tour_length(shortcut(MG, n), D)

def double_tree(D):
    n = len(D); G = nx.complete_graph(n)
    for u, v in G.edges: G[u][v]["weight"] = D[u][v]
    T = nx.minimum_spanning_tree(G)
    MG = nx.MultiGraph(T); MG.add_edges_from(T.edges)
    return tour_length(shortcut(MG, n), D)

t0 = time.time(); rc, rd = [], []
for trial in range(60):
    P = rng.random((11, 2)); D = np.linalg.norm(P[:, None] - P[None], axis=2).tolist()
    opt = held_karp(D)
    rc.append(christofides(D) / opt); rd.append(double_tree(D) / opt)
print(f"60 random sets of 11 points ({time.time()-t0:.0f} s)")
print(f"Christofides / OPT: mean {np.mean(rc):.3f}, worst {max(rc):.3f}   (guarantee 1.5)")
print(f"double tree  / OPT: mean {np.mean(rd):.3f}, worst {max(rd):.3f}   (guarantee 2)")
print(f"instances where the double tree beats Christofides: {sum(d < c - 1e-9 for c, d in zip(rc, rd))} of 60")
plt.figure(figsize=(6, 3))
plt.hist([rc, rd], bins=np.linspace(1, 1.6, 25), label=["Christofides", "double tree"])
plt.xlabel("tour / optimum"); plt.legend(); plt.tight_layout(); plt.show()

Both algorithms behave much better than their guarantees on random points: the guarantees are about the worst metric, and the plane is a kind place. Christofides wins on average, but not on every instance: the shortcuts depend on the Euler tour chosen, which the proof does not control. The Held–Karp program costs $2^{n}n^2$ operations, which already limits us to about a dozen cities in Python; the approximation algorithms run in polynomial time on thousands.

### P12.4 Rounding values

In [ ]:
def knap_by_value(w, v, W):
    # least weight for each total value (Exercise 5.5); returns best value and a subset
    V = sum(v); INF = float("inf")
    least = [0] + [INF] * V; choice = [[] for _ in range(V + 1)]
    for i in range(len(w)):
        for s in range(V, v[i] - 1, -1):
            if least[s - v[i]] + w[i] < least[s]:
                least[s] = least[s - v[i]] + w[i]; choice[s] = choice[s - v[i]] + [i]
    s = max(s for s in range(V + 1) if least[s] <= W)
    return s, choice[s]

def fptas(w, v, W, eps):
    K = eps * max(v) / len(v)
    vr = [int(x // K) for x in v]
    _, S = knap_by_value(w, vr, W)
    return sum(v[i] for i in S)

def greedy_knap(w, v, W):
    order = sorted(range(len(w)), key=lambda i: -v[i] / w[i]); tot = wt = 0
    for i in order:
        if wt + w[i] <= W: wt += w[i]; tot += v[i]
    return max(tot, max(v))

n = 60
w = [random.randint(10, 100) for _ in range(n)]; v = [random.randint(1000, 9000) for _ in range(n)]; W = sum(w) // 3
t0 = time.time(); opt, _ = knap_by_value(w, v, W); t_exact = time.time() - t0
print(f"exact: value {opt}, {t_exact:.2f} s")
for eps in [0.5, 0.2, 0.1, 0.05]:
    t0 = time.time(); val = fptas(w, v, W, eps)
    print(f"eps = {eps:4}: value {val} = {val/opt:.4f} OPT (guarantee {1-eps:.2f}), {time.time()-t0:.3f} s")
print(f"greedy of Exercise 12.5: {greedy_knap(w, v, W)/opt:.4f} OPT (guarantee 0.5)")

With large values the exact dynamic program over values is slow, while the rounded programs are fast and far more accurate than their guarantee: the error $nK=\varepsilon v_{\max}$ of the proof is a worst case that the rounding errors, which mostly cancel, rarely reach. The greedy algorithm of Exercise 12.5 is excellent here too; its factor $\frac12$ only bites when one large item does not fit after the small dense ones.

### P12.5 Three roads to MAX-SAT

In [ ]:
def random_formula(nv, m):
    F = []
    for _ in range(m):
        k = random.choice([1, 2, 2, 3, 3, 4])
        F.append([(x + 1) * random.choice([1, -1]) for x in random.sample(range(nv), k)])
    return F

def satisfied(F, a):     # a: array of 0/1, a[i] for variable i+1
    return sum(any((a[abs(l) - 1] == 1) == (l > 0) for l in c) for c in F)

def exact(F, nv):
    A = np.array(list(itertools.product([0, 1], repeat=nv)))
    tot = np.zeros(len(A), dtype=int)
    for c in F:
        sat = np.zeros(len(A), dtype=bool)
        for l in c: sat |= (A[:, abs(l) - 1] == (1 if l > 0 else 0))
        tot += sat
    return tot.max()

def expected(F, p):      # expected number satisfied when x_i = 1 with probability p_i
    return sum(1 - np.prod([1 - p[abs(l) - 1] if l > 0 else p[abs(l) - 1] for l in c]) for c in F)

def derandomise(F, nv, p):
    p = list(p)
    for i in range(nv):
        p[i] = 1.0; e1 = expected(F, p)
        p[i] = 0.0; e0 = expected(F, p)
        p[i] = 1.0 if e1 >= e0 else 0.0
    return satisfied(F, np.array(p, dtype=int))

def lp_maxsat(F, nv):
    m = len(F)                                  # variables y_1..y_nv, z_1..z_m; maximise sum z
    c = np.r_[np.zeros(nv), -np.ones(m)]
    A = np.zeros((m, nv + m)); b = np.zeros(m)
    for j, cl in enumerate(F):                  # z_j <= sum_{pos} y + sum_{neg} (1 - y)
        A[j, nv + j] = 1
        for l in cl:
            if l > 0: A[j, l - 1] -= 1
            else: A[j, -l - 1] += 1; b[j] += 1
    r = linprog(c, A_ub=A, b_ub=b, bounds=(0, 1), method="highs")
    return r.x[:nv], -r.fun

res = []
for trial in range(40):
    nv, m = 14, 70
    F = random_formula(nv, m); opt = exact(F, nv)
    y, lp = lp_maxsat(F, nv)
    half = expected(F, [0.5] * nv); rnd = expected(F, y)
    res.append((half / opt, derandomise(F, nv, [0.5] * nv) / opt, rnd / opt,
                (half + rnd) / 2 / opt, derandomise(F, nv, y) / opt, opt / lp))
res = np.array(res)
names = ["E[random 1/2]", "derandomised 1/2", "E[LP rounding]", "E[best of two coins]", "derandomised LP", "OPT / LP"]
for k, nm in enumerate(names):
    print(f"{nm:22s}: mean {res[:,k].mean():.3f}, worst {res[:,k].min():.3f}")

The coin and the rounded linear program each satisfy, in expectation, well over the $\frac12$ and $1-\frac1e$ of their analyses; their average, which is what the proof of Theorem 12.11 bounds by $\frac34$, does too. The method of conditional expectations turns every expectation into an actual assignment that is at least as good, and from the fractional solution it lands within a few per cent of the optimum. The relaxation itself is close to tight on these formulas: the ratio $\mathrm{OPT}/\mathrm{LP}$ stays near $1$, while Exercise 12.7 builds formulas where it is $\frac34$.

### P12.6 A random hyperplane

In [ ]:
th = np.linspace(1e-6, np.pi, 200001)
ratio = (th / np.pi) / ((1 - np.cos(th)) / 2)
print(f"alpha_GW = min (theta/pi) / ((1-cos theta)/2) = {ratio.min():.6f} at theta = {th[ratio.argmin()]:.4f}")

for theta in [0.5, 1.5, 2.5]:
    vi = np.zeros(20); vi[0] = 1; vj = np.zeros(20); vj[0], vj[1] = math.cos(theta), math.sin(theta)
    R = rng.standard_normal((200000, 20))
    print(f"theta = {theta}: P(separated) = {np.mean(np.sign(R @ vi) != np.sign(R @ vj)):.4f}, theta/pi = {theta/np.pi:.4f}")

def sdp_maxcut(Wm, d=None, iters=300):
    n = len(Wm); d = d or n
    V = rng.standard_normal((n, d)); V /= np.linalg.norm(V, axis=1, keepdims=True)
    for _ in range(iters):
        for i in range(n):            # best v_i with the others fixed: opposite to sum_j w_ij v_j
            s = Wm[i] @ V
            if np.linalg.norm(s) > 1e-12: V[i] = -s / np.linalg.norm(s)
    return V, np.sum(np.triu(Wm, 1) * (1 - V @ V.T)) / 2

def cut_value(Wm, s): return np.sum(np.triu(Wm, 1) * (1 - np.outer(s, s))) / 2

def brute_maxcut(Wm):
    n = len(Wm)
    S = np.array(list(itertools.product([1, -1], repeat=n - 1)))
    S = np.c_[np.ones(len(S)), S]
    return ((np.sum(Wm) - np.einsum("ki,ij,kj->k", S, Wm, S)) / 4).max()

out = []
for trial in range(25):
    n = 16
    Wm = np.triu((rng.random((n, n)) < 0.35) * rng.integers(1, 5, (n, n)), 1); Wm = Wm + Wm.T
    V, sdp = sdp_maxcut(Wm)
    L = np.diag(Wm.sum(1)) - Wm; upper = n / 4 * np.linalg.eigvalsh(L).max()
    opt = brute_maxcut(Wm)
    cuts = [cut_value(Wm, np.sign(V @ rng.standard_normal(n))) for _ in range(100)]
    out.append((opt / sdp, np.mean(cuts) / sdp, np.max(cuts) / opt, sdp <= upper + 1e-6, opt <= sdp + 1e-6))
out = np.array(out)
print(f"25 random weighted graphs on 16 vertices")
print(f"OPT <= SDP <= (n/4) lambda_max in every case: {bool(out[:,3].all() and out[:,4].all())}")
print(f"OPT / SDP: mean {out[:,0].mean():.3f}, worst {out[:,0].min():.3f}")
print(f"mean hyperplane cut / SDP: mean {out[:,1].mean():.3f}, worst {out[:,1].min():.3f}  (Theorem 12.14: at least 0.878)")
print(f"best of 100 hyperplanes / OPT: mean {out[:,2].mean():.4f}, worst {out[:,2].min():.4f}")

C5 = np.zeros((5, 5))
for i in range(5): C5[i, (i + 1) % 5] = C5[(i + 1) % 5, i] = 1
V, sdp = sdp_maxcut(C5, d=2, iters=2000)
print(f"pentagon: SDP = {sdp:.4f} (5(1-cos(4pi/5))/2 = {5*(1-math.cos(4*math.pi/5))/2:.4f}), OPT = {brute_maxcut(C5):.0f}, ratio {4/sdp:.4f}")
ang = np.degrees(np.arctan2(V[:, 1], V[:, 0])) % 360
print("angles of the five vectors:", np.round(ang - ang[0]) % 360)

The minimum over $\theta$ is the constant $0.878567\ldots$ of Lemma 12.13, reached at $\theta\approx2.33$, and the simulated separation probabilities match $\theta/\pi$ to three decimals. Coordinate ascent finds a feasible point of the semidefinite program, so a lower bound for its value; the sandwich $\mathrm{OPT}\le\mathrm{SDP}\le\frac n4\lambda_{\max}(L)$ held every time, which means that our ascent never stopped below the true optimum $\mathrm{OPT}$. A single random hyperplane gets on average far more than $0.878$ times the relaxation, and the best of a hundred found the maximum cut in every one of our graphs. On the pentagon the five vectors spread at multiples of $144^\circ$, a star, and the gap $4/4.523=0.884$ of Exercise 12.8 appears.

### P12.7 Skis and pages

In [ ]:
r = 1.0
for B in [2, 5, 10, 20]:
    worst = max((min(d, B - 1) * r + (B if d >= B else 0)) / min(d, B) for d in range(1, 5 * B))
    print(f"B = {B:2d}: buy on day B, worst ratio over season lengths {worst:.3f} = 2 - 1/B = {2 - 1/B:.3f}")

def lru(seq, k):
    cache, f = [], 0
    for p in seq:
        if p in cache: cache.remove(p)
        else:
            f += 1
            if len(cache) == k: cache.pop(0)
        cache.append(p)
    return f

def fifo(seq, k):
    cache, f = [], 0
    for p in seq:
        if p not in cache:
            f += 1
            if len(cache) == k: cache.pop(0)
            cache.append(p)
    return f

def belady(seq, k):
    nxt, last = [0] * len(seq), {}
    for t in range(len(seq) - 1, -1, -1):
        nxt[t] = last.get(seq[t], math.inf); last[seq[t]] = t
    cache, f = {}, 0                    # page -> time of its next request
    for t, p in enumerate(seq):
        if p not in cache:
            f += 1
            if len(cache) == k: del cache[max(cache, key=cache.get)]
        cache[p] = nxt[t]
    return f

k, N = 8, 20000
zipf = rng.zipf(1.3, N) % 50
loop = [t % (k + 1) for t in range(N)]
seqs = {"uniform on 20 pages": list(rng.integers(0, 20, N)), "Zipf on 50 pages": list(zipf), f"cycle of {k+1} pages": loop}
for name, s in seqs.items():
    o = belady(s, k)
    print(f"{name:20s}: OPT {o:6d}   LRU {lru(s, k):6d} ({lru(s, k)/o:.2f} OPT)   FIFO {fifo(s, k):6d} ({fifo(s, k)/o:.2f} OPT)")

def adversary(alg, k, N):        # always request the page missing from alg's cache (k+1 pages)
    seq = []
    for _ in range(N):
        cache = simulate_cache(alg, seq, k)
        seq.append(next(p for p in range(k + 1) if p not in cache))
    return seq

def simulate_cache(alg, seq, k):
    cache = []
    for p in seq:
        if alg == "LRU" and p in cache: cache.remove(p); cache.append(p); continue
        if p in cache: continue
        if len(cache) == k: cache.pop(0)
        cache.append(p)
    return cache

for alg, f in [("LRU", lru), ("FIFO", fifo)]:
    s = adversary(alg, k, 400)
    print(f"adversary against {alg}: {alg} faults {f(s, k)}; OPT faults {belady(s, k)}, of which {k} fill the empty cache, so {belady(s, k) - k} <= ceil(N/k) = {math.ceil(400/k)}")

The worst season for the rule "buy on day $B$" is the one that ends just after buying, and the ratio is exactly $2-\frac1B$. For paging, the cycle on $k+1$ pages is the nightmare of the second part of Theorem 12.16: LRU and FIFO evict precisely the page that comes next and fault on every request, while Belady's rule faults once per $k$ requests or so. On random and Zipf-like sequences, which have locality or none, both online rules stay within a small factor of the optimum, nowhere near $k$. The adversary, who reads the cache, forces $N$ faults; the optimum, after the $k$ faults that fill an empty cache (the theorem starts both with the same full cache), makes at most $\lceil N/k\rceil$, as in the proof.

### P12.8 Experts

In [ ]:
def hedge(losses, eta):
    T, n = losses.shape; w = np.ones(n); total = 0.0
    for t in range(T):
        p = w / w.sum(); total += p @ losses[t]; w *= np.exp(-eta * losses[t])
    return total

T, n = 5000, 20
bound = 2 * math.sqrt(T * math.log(n)) + math.log(n); eta = math.sqrt(math.log(n) / T)
cases = {}
mu = rng.uniform(0.3, 0.7, n); cases["iid, different means"] = (rng.random((T, n)) < mu).astype(float)
X = np.ones((T, n)); X[: T // 2, 0] = 0; X[2 * T // 5 :, 1] = 0; cases["leader changes at 0.4T"] = X
Y = np.zeros((T, n)); Y[::2, : n // 2] = 1; Y[1::2, n // 2 :] = 1; cases["alternating halves"] = Y
for name, Lm in cases.items():
    reg = hedge(Lm, eta) - Lm.sum(0).min()
    print(f"{name:22s}: regret {reg:7.1f}, bound {bound:.1f}")

def follow_leader(losses):
    total, cum = 0.0, np.zeros(losses.shape[1])
    for t in range(len(losses)):
        i = int(np.argmin(cum)); total += losses[t, i]; cum += losses[t]
    return total
Z = np.zeros((T, 2)); Z[0, 0] = 0.5                   # then the leader is always the expert about to lose
for t in range(1, T): Z[t, t % 2] = 1
print(f"follow the leader on 2 experts: loses {follow_leader(Z):.0f}, best expert {Z.sum(0).min():.1f}, Hedge {hedge(Z, math.sqrt(math.log(2)/T)):.0f}")

reg_T = []
Ts = [100, 300, 1000, 3000, 10000]
for T_ in Ts:
    Lm = (rng.random((T_, n)) < rng.uniform(0.4, 0.6, n)).astype(float)
    reg_T.append(hedge(Lm, math.sqrt(math.log(n) / T_)) - Lm.sum(0).min())
plt.figure(figsize=(5, 3))
plt.loglog(Ts, reg_T, "o-", label="regret of Hedge")
plt.loglog(Ts, [2 * math.sqrt(t * math.log(n)) + math.log(n) for t in Ts], "--", label=r"$2\sqrt{T\ln n}+\ln n$")
plt.xlabel("T"); plt.legend(); plt.tight_layout(); plt.show()

A = rng.integers(-3, 4, (4, 5)).astype(float)        # row player maximises x^T A y
m_, n_ = A.shape                                      # value by linear programming: max v, A^T x >= v, sum x = 1
c = np.r_[np.zeros(m_), -1]
r = linprog(c, A_ub=np.c_[-A.T, np.ones(n_)], b_ub=np.zeros(n_), A_eq=[np.r_[np.ones(m_), 0]], b_eq=[1],
            bounds=[(0, None)] * m_ + [(None, None)])
value = -r.fun
T = 20000; eta = math.sqrt(math.log(5) / T); s = A.max() - A.min()
wr, wc, avg = np.ones(m_), np.ones(n_), 0.0
for t in range(T):
    x, y = wr / wr.sum(), wc / wc.sum(); avg += x @ A @ y / T
    wr *= np.exp(-eta * (A.max() - A @ y) / s)        # the row player's losses, rescaled into [0, 1]
    wc *= np.exp(-eta * (A.T @ x - A.min()) / s)
print(f"game value by LP {value:.4f}; average payoff of two Hedge players {avg:.4f}")

The regret of Hedge stays below its bound on every sequence. When the leader changes (expert 0 is perfect for the first half, expert 1 from $0.4T$ on and is the best overall), the weights take a while to forget the past, which is what the $\sqrt T$ of the bound pays for. Follow-the-leader, a deterministic rule, is fooled every single day by the alternating losses of Exercise 12.10, and loses about $T$ while the best expert loses $T/2$; the randomised Hedge loses about $T/2$ as well. The log-log plot shows the regret growing like $\sqrt T$, parallel to the bound. Finally, two Hedge players who only react to each other's mixed strategies reach, on average, the value of the game computed by linear programming: this is the proof of von Neumann's minimax theorem of Exercise 7.7, run as a program.